# Delivery planner checks: Larch Road Health Centre

Is the information planned in the MIDP arriving on time, at the right status, with metadata that matches the plan and the client's reference data?

**Inputs** (in `delivery_planner/data/`): the client's reference library, the MIDP, the P6 programme (baseline and the 1 May 2027 update), and a ProjectWise export with one row per version.

**What this notebook does**
1. Gets the code from GitHub (or from an uploaded zip).
2. Loads the five inputs through the adapters.
3. Runs 25 checks on the *perfect* data: the answer should be zero issues.
4. Breaks the data with 28 deliberate faults, one at a time, and scores whether the right check catches each one.
5. Runs all 28 faults together and shows the delivery status register and the issues list.
6. Saves an Excel report you can download.

Run it top to bottom: **Runtime → Run all**, or one cell at a time with Shift+Enter.

## 1. Settings

If your GitHub repository is **public**, type its name below (for example `fred/cde-dq-checker`).
If it is **private**, leave it blank: the next cell will ask you to upload the zip instead
(on GitHub: switch to the `iteration-3` branch → **Code → Download ZIP**).

In [ ]:
GITHUB_REPO = ""          # e.g. "your-user/your-repo"; leave blank to upload a zip
BRANCH = "iteration-3"

## 2. Get the code

In [ ]:
import os, sys, glob, subprocess, zipfile

def find_planner(root="."):
    hits = [p for p in glob.glob(os.path.join(root, "**", "delivery_planner", "checks"), recursive=True)]
    return os.path.dirname(hits[0]) if hits else None

planner = find_planner("/content") if os.path.isdir("/content") else find_planner(".")
if planner is None and GITHUB_REPO:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, f"https://github.com/{GITHUB_REPO}.git", "repo"], check=True)
    planner = find_planner("repo")
if planner is None:
    from google.colab import files            # private repo: upload the GitHub zip
    for name in files.upload():
        zipfile.ZipFile(name).extractall("repo")
    planner = find_planner("repo")

planner = os.path.abspath(planner)
os.chdir(planner)
if planner not in sys.path:
    sys.path.insert(0, planner)
print("Working in", planner)

## 3. Load the inputs

`load_inputs` reads each file through its adapter and prepares the tables the checks use.
The programme can be the `.xer` or the `.csv`: the adapter returns the same table either way.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 80)
from checks.run import PERFECT, run_checks, register, catalogue
from checks.common import load_inputs

d = load_inputs(**PERFECT)
print("Data date:", d["data_date"].strftime("%d %b %Y"))
print("Reference library:", len(d["library"]), "codes")
print("MIDP:", len(d["midp"]), "rows,", d["midp"].container_name.nunique(), "containers")
print("Programme:", len(d["update"]), "activities")
print("CDE export:", len(d["cde"]), "versions of", d["cde"].container_name.nunique(), "containers")
d["midp"].head()

In [ ]:
d["cde"].head()

## 4. The checks

25 checks in five groups, each mapped to a DAMA data quality dimension.
R = reference data, P = programme link, C = completeness, T = timeliness, M = metadata match.

In [ ]:
catalogue()

## 5. Run the checks on the perfect data

Everything in the perfect data was delivered exactly as planned, so this must give **zero issues**.
If it doesn't, a check is wrong (a false alarm), not the data.

In [ ]:
issues = run_checks(d)
reg = register(d, issues)
print(len(issues), "issues")
reg.delivery_status.value_counts()

### The delivery status register

One row per MIDP row. Due dates come from the linked P6 activity: baseline (contractual) and forecast (current update).
`required_reached_on` is the first time the CDE held the container at the required suitability or better.

In [ ]:
reg

## 6. Break it: one fault at a time

Each of the 28 faults edits the raw input files, the way a real mistake would appear in an export.
A fault is **caught** when every check it is aimed at fires on the right container.
Any other check firing is a **false alarm**.

In [ ]:
from faults.score import score
scores = score()
print("Caught:", (scores.caught == "yes").sum(), "of", len(scores),
      "| faults with false alarms:", (scores.false_alarms != "").sum())
scores[["fault_id", "fault", "expected", "fired", "caught", "false_alarms"]]

## 7. All 28 faults at once

This is what a reviewer would see on a bad day. The broken input files are written to `data/broken/` so you can open them.

In [ ]:
from faults.score import broken
d_b, issues_b, reg_b = broken()
print(len(issues_b), "issues:", (issues_b.severity == "error").sum(), "errors,", (issues_b.severity == "warning").sum(), "warnings")
reg_b.delivery_status.value_counts()

In [ ]:
issues_b[["check_id", "severity", "container", "message"]]

In [ ]:
reg_b[reg_b.result != "Clean"][["midp_id", "container", "exchange", "due_baseline", "due_forecast",
                                "required_reached_on", "delivery_status", "errors", "warnings"]]

## 8. Save the report

In [ ]:
from faults.score import main
main()                                   # writes outputs/*.csv and outputs/delivery_check_report.xlsx
try:
    from google.colab import files
    files.download("outputs/delivery_check_report.xlsx")
except ImportError:
    print("Report saved to outputs/delivery_check_report.xlsx")